# Qualifire — End-to-End Daily Pipeline Demo

Three datasets validated by two YAML configs, simulated over **14 daily
partitions** so drift / forecast / pattern checks accumulate real history.

| Dataset | Source | Collections | Validations |
|---|---|---|---|
| `retail.sales_fact` | seeded ~500 rows/day | `recency`, `aggregation`, `profiling`, `sample` | SLO + threshold + drift + profiling-threshold + pattern |
| `marketing.campaign_events` | seeded ~200 rows/day | `aggregation` (×2 with `dimensions`), `sample` | per-channel threshold + forecast + pattern |
| `revenue.daily_summary` | WAP roll-up of D1+D2 | `aggregation` (on staging temp view) | threshold floor + day-over-day rate-of-change |

**Notebook layout** (each phase is its own cell, easy to re-run in
isolation):

| Phase | What runs | Qualifire calls |
|---|---|---|
| Setup | Spark, Qualifire init, render YAMLs, register `inbox` notifier | — |
| **Phase A — Seed** | `seed_sales(d)` and `seed_campaigns(d)` for all 14 partitions | none |
| **Phase B — Validate sources** | One `qf.run_config(SOURCES_YAML)` per partition (D1+D2 together) | 14 calls |
| **Phase C — WAP D3** | One `qf.run_config(WAP_YAML)` per partition (write→audit→publish/rollback) | 14 calls |
| Phase D — Dashboard | `qf.health_report(...)` + `qf.interactive_dashboard(...)` | — |
| Cleanup | Stop Spark, remove workdir | — |

**What to expect**:
- Days 1–10: stable seeded baseline → mostly **PASS**. Cold-start
  WARNs are normal in the first 1–3 partitions (forecast / pattern
  need history to fit).
- Days 11–12: 1.5× mean ramp → drift / forecast may WARN; profiling
  may WARN as `null_pid_rate` ticks up.
- Days 13–14: 2.8× ramp → drift fires **ERROR**, profiling may
  **ERROR** on `product_id.null_pct`, pattern AUC climbs.
- Each ERROR also lands in the `inbox` notifier — printed at the end
  of each phase loop iteration.
- Pattern / shape validations now include their top-3 SHAP features
  in the message column (e.g. `AUC=0.84 · top: region_us(+0.32),
  amount(+0.18), …`), so the *why* shows up in the dashboard table.

YAML configs:
- [`configs/end_to_end_sources.yaml`](configs/end_to_end_sources.yaml)
- [`configs/end_to_end_wap.yaml`](configs/end_to_end_wap.yaml)


## 1. Bootstrap — sys.path / JAVA_HOME / Spark

Same hardening as `notebook_overall.ipynb` (Downloads-folder scrub +
Java 8/11/17 discovery + per-run Hive metastore). If anything here
fails, run [`pyspark_setup.ipynb`](pyspark_setup.ipynb) first.

In [ ]:
import os, sys
before = len(sys.path)
sys.path[:] = [p for p in sys.path if '/Downloads/' not in p]
os.environ.pop('PYTHONPATH', None)
spark_home = os.environ.get('SPARK_HOME', '')
if '/Downloads/' in spark_home:
    print(f'unset SPARK_HOME (was {spark_home!r}; under ~/Downloads)')
    os.environ.pop('SPARK_HOME', None)
print(f'sys.path: kept {len(sys.path)}/{before} entries; PYTHONPATH unset')


In [ ]:
import os, re, glob, subprocess
ACCEPTABLE = (8, 11, 17)

def _major(java_bin):
    try:
        out = subprocess.check_output([java_bin, '-version'], stderr=subprocess.STDOUT, text=True)
    except Exception:
        return None
    m = re.search(r'version "([^"]+)"', out)
    if not m: return None
    parts = m.group(1).split('.')
    return 8 if parts[:2] == ['1', '8'] else int(parts[0])

def _discover_homes():
    homes, seen = [], set()
    def _add(h):
        if h and h not in seen and os.path.isdir(h):
            seen.add(h); homes.append(h)
    patterns = [
        '/Library/Java/JavaVirtualMachines/*/Contents/Home',
        os.path.expanduser('~/Library/Java/JavaVirtualMachines/*/Contents/Home'),
        '/opt/homebrew/Cellar/openjdk*/*/libexec/openjdk.jdk/Contents/Home',
        '/usr/local/Cellar/openjdk*/*/libexec/openjdk.jdk/Contents/Home',
        '/opt/homebrew/opt/openjdk*/libexec/openjdk.jdk/Contents/Home',
        os.path.expanduser('~/.sdkman/candidates/java/*'),
    ]
    for pat in patterns:
        for h in glob.glob(pat): _add(h)
    try:
        listing = subprocess.check_output(
            ['/usr/libexec/java_home', '-V'], text=True, stderr=subprocess.STDOUT)
        for line in listing.splitlines():
            m = re.search(r'\s(\S+/Contents/Home)\s*$', line)
            if m: _add(m.group(1))
    except Exception:
        pass
    return homes

override = os.environ.get('QF_JAVA_HOME', '').strip()
homes = [override] if override else _discover_homes()
acceptable = []
for h in homes:
    if 'JavaAppletPlugin' in h: continue
    java_bin = os.path.join(h, 'bin', 'java')
    if not os.path.isfile(java_bin): continue
    mj = _major(java_bin)
    if mj in ACCEPTABLE:
        acceptable.append((h, mj))
acceptable.sort(key=lambda hm: ACCEPTABLE.index(hm[1]))
if not acceptable:
    raise RuntimeError('No usable Java 8/11/17 JDK. Set QF_JAVA_HOME=/path/to/jdk.')
java_home, mj = acceptable[0]
os.environ['JAVA_HOME'] = java_home
os.environ['PATH'] = f"{java_home}/bin:" + os.environ.get('PATH', '')
print(f'==> JAVA_HOME = {java_home}  (Java {mj})')


In [ ]:
import tempfile
from pathlib import Path

os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

WORK_DIR = Path(tempfile.mkdtemp(prefix='qualifire_e2e_'))
WAREHOUSE = WORK_DIR / 'warehouse'
SYSTEM_DB = WORK_DIR / 'qualifire_history.sqlite'
REPORT_HTML = WORK_DIR / 'health_report.html'
INTERACTIVE_HTML = WORK_DIR / 'interactive_dashboard.html'

# Where the rendered (substituted) YAML configs go.
CONFIG_DIR = WORK_DIR / 'configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Source YAML templates (committed alongside this notebook). Walk
# parents looking for tests/manual/configs so this works whether the
# notebook is launched from tests/manual or the project root.
_candidates = [Path.cwd() / 'configs',
               Path.cwd() / 'tests' / 'manual' / 'configs']
for _p in [Path.cwd(), *Path.cwd().parents]:
    _candidates.append(_p / 'tests' / 'manual' / 'configs')
TEMPLATE_DIR = next(
    (p for p in _candidates if (p / 'end_to_end_sources.yaml').exists()),
    None,
)
if TEMPLATE_DIR is None:
    raise FileNotFoundError(
        f'configs/end_to_end_sources.yaml not found relative to {Path.cwd()}'
    )

print('Workdir   :', WORK_DIR)
print('Warehouse :', WAREHOUSE)
print('System DB :', SYSTEM_DB)
print('Templates :', TEMPLATE_DIR)


In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

if SparkContext._gateway is not None:
    try: SparkContext._gateway.shutdown()
    except Exception: pass
    SparkContext._gateway = None
    SparkContext._jvm = None
    SparkContext._active_spark_context = None

metastore_url = f"jdbc:derby:;databaseName={WORK_DIR}/metastore_db;create=true"
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('qualifire-e2e')
    .config('spark.sql.warehouse.dir', str(WAREHOUSE))
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
    .config('javax.jdo.option.ConnectionURL', metastore_url)
    .config('spark.driver.extraJavaOptions', f'-Dderby.system.home={WORK_DIR}')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
runtime = spark.sparkContext._jvm.System.getProperty('java.version')
assert runtime.startswith(('1.8.', '11.', '17.')), f'Spark on bad Java {runtime}'

for db in ('retail', 'marketing', 'revenue'):
    spark.sql(f'CREATE DATABASE IF NOT EXISTS {db}')
for tbl in ('retail.sales_fact', 'marketing.campaign_events', 'revenue.daily_summary'):
    spark.sql(f'DROP TABLE IF EXISTS {tbl}')

# Pre-create the WAP target table so the very first day's
# ``INSERT INTO revenue.daily_summary SELECT * FROM <staging>``
# has somewhere to land. Schema mirrors the WAP write_sql output.
spark.sql(
    "CREATE TABLE revenue.daily_summary ("
    "  summary_date STRING,"
    "  total_sales_amount DOUBLE,"
    "  total_sales_rows BIGINT,"
    "  total_conversion_value DOUBLE,"
    "  total_sessions BIGINT,"
    "  combined_revenue DOUBLE,"
    "  published_at TIMESTAMP"
    ") USING parquet"
)
print(f'Spark up on Java {runtime}; databases + WAP target ready.')


In [ ]:
from qualifire.api import Qualifire
from qualifire.backends.spark_backend import SparkBackend

qf = Qualifire(
    backend=SparkBackend(spark),
    system_table=str(SYSTEM_DB),
    system_table_backend='sqlite',
    owner='data-quality-demo',
    bu='demo',
)
qf


## 2. Render YAML configs

`load_config` doesn't expand Jinja over scalar fields like
`system_table`, so we substitute `{{ system_db }}` ourselves and
drop the rendered copies into the workdir. The notebook then runs
those rendered configs.

In [ ]:
SOURCES_YAML = CONFIG_DIR / 'end_to_end_sources.yaml'
WAP_YAML     = CONFIG_DIR / 'end_to_end_wap.yaml'

for src_name, dst in [
    ('end_to_end_sources.yaml', SOURCES_YAML),
    ('end_to_end_wap.yaml',     WAP_YAML),
]:
    body = (TEMPLATE_DIR / src_name).read_text(encoding='utf-8')
    dst.write_text(body.replace('{{ system_db }}', str(SYSTEM_DB)), encoding='utf-8')
    print(f'  rendered {src_name:30s} -> {dst}')


## 3. CapturingNotifier — print notifications instead of sending

Same idiom as `notebook_overall.ipynb`: a tiny in-process notifier
registered as channel `inbox`. Both YAMLs route their `notify:` blocks
to `inbox`, so every alert that fires lands in `inbox.sent`.

In [ ]:
from qualifire.notification.base import Notifier
from qualifire.core.models import NotificationResult

class CapturingNotifier(Notifier):
    def __init__(self):
        self.sent = []

    @property
    def channel_name(self) -> str:
        return 'inbox'

    def send(self, dataset_result, severity, owner, bu, datasets=None):
        message = self._format_message(dataset_result, severity, owner, bu, datasets)
        self.sent.append({
            'severity': severity.value,
            'dataset': dataset_result.dataset_name,
            'message': message,
        })
        return NotificationResult(
            channel=self.channel_name, severity=severity,
            status='sent', message='captured locally',
        )

inbox = CapturingNotifier()
qf.register_notifier('inbox', inbox)
print('inbox notifier registered.')


## 4. Seeders — D1 (sales_fact) and D2 (campaign_events)

Each call appends one partition's worth of rows to a managed Spark table:

| Table | Schema (key cols) | Per-partition shape |
|---|---|---|
| `retail.sales_fact` | `sale_id, product_id, region, amount, updated_at, sale_date` | 500 rows · region uniformly drawn from {us, uk, jp, de} · `amount ~ Gauss(mean_amount, σ=0.15·mean)` · `product_id` is NULL with probability `null_pid_rate` |
| `marketing.campaign_events` | `event_id, channel, sessions, conversion_value, conversions, event_ts, event_date` | 200 rows · channel uniformly drawn from {email, paid, organic, social} · `sessions ~ Gauss(mean_sessions, σ=0.2·mean)` |

`updated_at` is wall-clock `now()` (so the SLO check sees fresh data even
on backfilled partitions). `sale_date` / `event_date` are the partition
identity that drift / forecast / pattern anchor their history reads on.

The loop varies `mean_amount`, `mean_sessions`, and `null_pid_rate` by day
(`_scenario(day_idx)` further down) so the synthetic series has enough
movement for the history-backed validators to detect.

In [ ]:
import random
from datetime import datetime, timedelta, date
from pyspark.sql import Row

random.seed(42)
REGIONS  = ['us', 'uk', 'jp', 'de']
CHANNELS = ['email', 'paid', 'organic', 'social']

def seed_sales(sale_date: date, n_rows: int = 500, mean_amount: float = 50.0,
               null_pid_rate: float = 0.0, overwrite_table: bool = False):
    """Append one day's partition to retail.sales_fact.

    ``updated_at`` is stamped with wall-clock now so the SLO check
    sees fresh data even when we backfill a 14-day-old partition.
    The partition column ``sale_date`` keeps the partition identity
    that drift / forecast / pattern anchor their history reads on.
    """
    now = datetime.now()
    rows = []
    for i in range(n_rows):
        amt = max(1.0, random.gauss(mean_amount, mean_amount * 0.15))
        pid = None if random.random() < null_pid_rate else random.randint(1, 50)
        rows.append(Row(
            sale_id=f'{sale_date.isoformat()}-{i}',
            product_id=pid,
            region=random.choice(REGIONS),
            amount=float(round(amt, 2)),
            updated_at=now,
            sale_date=sale_date.isoformat(),
        ))
    df = spark.createDataFrame(rows)
    mode = 'overwrite' if overwrite_table else 'append'
    (df.write.mode(mode).partitionBy('sale_date')
       .format('parquet').saveAsTable('retail.sales_fact'))
    return n_rows

def seed_campaigns(event_date: date, n_rows: int = 200, mean_sessions: float = 30.0,
                   mean_conv_value: float = 25.0, overwrite_table: bool = False):
    """Append one day's partition to marketing.campaign_events."""
    rows = []
    for i in range(n_rows):
        sess = max(1, int(random.gauss(mean_sessions, mean_sessions * 0.2)))
        conv_val = max(0.0, random.gauss(mean_conv_value, mean_conv_value * 0.3))
        conversions = max(0, int(round(conv_val / 5.0)))
        rows.append(Row(
            event_id=f'{event_date.isoformat()}-{i}',
            channel=random.choice(CHANNELS),
            sessions=sess,
            conversion_value=float(round(conv_val, 2)),
            conversions=conversions,
            event_ts=datetime.combine(event_date, datetime.min.time()) + timedelta(hours=8),
            event_date=event_date.isoformat(),
        ))
    df = spark.createDataFrame(rows)
    mode = 'overwrite' if overwrite_table else 'append'
    (df.write.mode(mode).partitionBy('event_date')
       .format('parquet').saveAsTable('marketing.campaign_events'))
    return n_rows


## Phase A — Seed all 14 daily partitions

Pure ETL: write `retail.sales_fact` and `marketing.campaign_events`
for every date P in `[today − 13, today]`. Qualifire is **not**
called here — we just stage the data so later phases can validate
and roll it up.

The per-day parameters come from `_scenario(day_idx)`:
- Days 0–9 (indices `< 10`): stable baseline — `mean_amount=50`,
  `mean_sessions=30`, `null_pid_rate=0`.
- Days 10–11: mid ramp — `mean_amount=75`, `mean_sessions=45`,
  `null_pid_rate=0.005`.
- Days 12–13: strong ramp — `mean_amount=140`, `mean_sessions=70`,
  `null_pid_rate=0.02`.

Day 0 of the loop uses `overwrite_table=True` so any leftover
table from a previous kernel run is dropped before the seed
begins; subsequent days append.

In [ ]:
from datetime import date, timedelta

TODAY = date.today()
P_START = TODAY - timedelta(days=13)             # 14 partitions inclusive
ALL_PARTITIONS = [P_START + timedelta(days=i) for i in range(14)]


def _scenario(day_idx: int):
    """Per-day seeding parameters. Stable for 10 days, then ramped
    to give drift / forecast / pattern detectable signal."""
    if day_idx < 10:
        return dict(mean_amount=50.0, mean_sessions=30.0, null_pid_rate=0.0)
    if day_idx < 12:
        return dict(mean_amount=75.0, mean_sessions=45.0, null_pid_rate=0.005)
    return dict(mean_amount=140.0, mean_sessions=70.0, null_pid_rate=0.02)


print(f'Seeding 14 partitions: {ALL_PARTITIONS[0]} → {ALL_PARTITIONS[-1]}')
for i, d in enumerate(ALL_PARTITIONS):
    sc = _scenario(i)
    n_d1 = seed_sales(
        d, n_rows=500, mean_amount=sc['mean_amount'],
        null_pid_rate=sc['null_pid_rate'], overwrite_table=(i == 0),
    )
    n_d2 = seed_campaigns(
        d, n_rows=200, mean_sessions=sc['mean_sessions'],
        overwrite_table=(i == 0),
    )
    flag = '  ramp' if i >= 10 else ''
    print(f"  {d}  D1={n_d1:4d}  D2={n_d2:4d}{flag}")

# Sanity-check the staged data — should see 14 distinct dates per table.
spark.sql(
    'SELECT COUNT(DISTINCT sale_date) AS days, COUNT(*) AS rows '
    'FROM retail.sales_fact'
).show()
spark.sql(
    'SELECT COUNT(DISTINCT event_date) AS days, COUNT(*) AS rows '
    'FROM marketing.campaign_events'
).show()


## Phase B — Validate D1 + D2 for every partition

One `qf.run_config(SOURCES_YAML, context={"ds": iso})` per partition.
Each call runs **both D1 and D2** validations together (the sources
YAML declares both datasets) — that's a single Qualifire invocation
with multi-dataset fan-out.

Why iterate in date order: drift / forecast / pattern read past
partitions from the system table. Earlier iterations write
`record_type='collection'` rows under their `partition_ts`; later
iterations' history-backed validators then have something to
compare against.

Per iteration we print:
- `[severity] dataset.validation` for every `ValidationResult`
  produced (one row per `metric × dimension`).
- New `inbox` notifications captured during this day, headline
  only.

Cold-start days (first ~3) typically WARN on forecast / pattern
because `history_count` and `past_dates` aren't fully satisfied
yet; that's by design.

In [ ]:
from qualifire.core.exceptions import QualifireValidationError


def _summary(qf_result):
    """Compact (severity, dataset, validation) tuples per ValidationResult."""
    out = []
    for ds in qf_result.datasets:
        for vr in ds.validation_results:
            out.append((vr.severity.value, ds.dataset_name, vr.validation_name))
    return out


sources_runs = []
for d in ALL_PARTITIONS:
    iso = d.isoformat()
    inbox_pre = len(inbox.sent)
    try:
        result = qf.run_config(str(SOURCES_YAML), context={'ds': iso})
        rows = _summary(result)
    except QualifireValidationError as e:
        rows = _summary(e.result)
    except Exception as e:
        rows = [('engine_error', '?', repr(e))]
    new_notifications = inbox.sent[inbox_pre:]
    sources_runs.append({'partition': iso, 'rows': rows,
                         'notifications': new_notifications})

    print(f"\n=== {iso}  (sources YAML — D1 + D2) ===")
    for sev, ds, vn in rows:
        print(f"  [{sev:>7}] {ds}.{vn}")
    if new_notifications:
        print(f"  --- {len(new_notifications)} notification(s) ---")
        for n in new_notifications:
            head = n['message'].splitlines()[0] if n['message'] else '(empty)'
            print(f"    [{n['severity']:>7}] {n['dataset']:<30} {head}")


## Phase C — WAP-publish D3 for every partition

One `qf.run_config(WAP_YAML, context={"ds": iso})` per partition. The
WAP YAML's lone dataset has a `wap:` block, so each call goes through
the full Write→Audit→Publish flow:

1. **WRITE** — `write_sql` joins the partition's slice of D1+D2 into
   a single rolled-up row, registered as a Spark temp view
   (`cache: true` keeps it off the Hive metastore).
2. **AUDIT** — threshold (`combined_revenue` floor + sanity counts)
   and drift (day-over-day `rate_of_change_pct`) run against that
   view; their results land in the system table.
3. **PUBLISH** — if both checks landed `< ERROR`, the staged row is
   `INSERT INTO`'d to `revenue.daily_summary`. Otherwise the staging
   view is dropped (rollback) and that day never reaches the
   target table.

Iterating in date order matters here too: the rate-of-change check
reads yesterday's published row from D3's own history.

In [ ]:
wap_runs = []
for d in ALL_PARTITIONS:
    iso = d.isoformat()
    inbox_pre = len(inbox.sent)
    try:
        result = qf.run_config(str(WAP_YAML), context={'ds': iso})
        rows = _summary(result)
    except QualifireValidationError as e:
        rows = _summary(e.result)
    except Exception as e:
        rows = [('engine_error', '?', repr(e))]
    new_notifications = inbox.sent[inbox_pre:]
    wap_runs.append({'partition': iso, 'rows': rows,
                     'notifications': new_notifications})

    print(f"\n=== {iso}  (WAP YAML — D3) ===")
    for sev, ds, vn in rows:
        print(f"  [{sev:>7}] {ds}.{vn}")
    if new_notifications:
        print(f"  --- {len(new_notifications)} notification(s) ---")
        for n in new_notifications:
            head = n['message'].splitlines()[0] if n['message'] else '(empty)'
            print(f"    [{n['severity']:>7}] {n['dataset']:<30} {head}")


## Inspect the WAP-published target table

`revenue.daily_summary` should now hold one row per **published**
partition. Days where Phase C's threshold or rate-of-change check
landed at ERROR were rolled back — those days have no row here.

In [ ]:
try:
    spark.sql(
        """
        SELECT summary_date,
               ROUND(total_sales_amount, 2)        AS sales_amt,
               total_sales_rows                    AS sales_rows,
               ROUND(total_conversion_value, 2)    AS conv_value,
               total_sessions                      AS sessions,
               ROUND(combined_revenue, 2)          AS combined_rev
          FROM revenue.daily_summary
         ORDER BY summary_date
        """
    ).show(truncate=False)
except Exception as e:
    print(f'revenue.daily_summary not queryable yet: {e!r}')


## Phase D — Dashboards

Two views over the system table:

1. **Static health report** (`qf.health_report(...)`) — per-type
   counts of pass / warn / error. The HTML render is also written
   to disk for sharing.
2. **Interactive dashboard** (`qf.interactive_dashboard(...)`) —
   per-dataset / per-validation drill-down, partition-level history
   chart, and a recent-validations table. Rendered both as a
   standalone file (`INTERACTIVE_HTML` printed below) and inline
   inside an iframe so its JS runs even in Jupyter front-ends that
   sandbox inline `<script>` tags. The "Time range (days)" picker
   filters by `partition_ts` (the data-domain timestamp), not
   `run_timestamp`. With the 14-day backfill all partitions were
   stamped within today's wall-clock window, so a `run_timestamp`
   filter would be useless; partition_ts gives you the data window
   you actually care about.

The interactive dashboard's message column for `pattern` and
`shape` validations now ends with the top-3 SHAP contributors —
e.g. `Pattern AUC: 0.84 (+/-0.04) · top: region_us(+0.32),
amount(+0.18), product_id(+0.11)` — so the **why** is visible
without leaving the table.

In [ ]:
report = qf.health_report(days=30, output_path=str(REPORT_HTML))
print('totals  :', report.total_checks, 'checks')
print('rates   : pass={:.1f}%  warn={:.1f}%  error={:.1f}%'.format(
    report.pass_rate, report.warning_rate, report.error_rate))
for row in report.by_check_type:
    print('   ', row)
print(f'HTML report at: {REPORT_HTML}  ({REPORT_HTML.stat().st_size:,} bytes)')


In [ ]:
from IPython.display import HTML, display

# Save a standalone copy first; then inline-render with
# ``as_iframe=False`` (scripts run directly in the notebook output,
# scrolling chains naturally to the next cell).
qf.interactive_dashboard(output_path=str(INTERACTIVE_HTML),
                         days=30, inline_plotly_js=True)
print(f'Standalone dashboard written to: {INTERACTIVE_HTML}')
print(f'  ({INTERACTIVE_HTML.stat().st_size:,} bytes — `open <path>` in a browser if the inline render is blank.)')

display(HTML(qf.interactive_dashboard(
    days=30, inline_plotly_js=True, as_iframe=False,
)))


## Cleanup (optional)

Stops Spark, removes the workdir, and sweeps any Derby / Hive
metastore turds left in CWD by an earlier crashed run.

In [ ]:
import shutil
spark.stop()
shutil.rmtree(WORK_DIR, ignore_errors=True)
for name in ('metastore_db', 'derby.log'):
    target = Path.cwd() / name
    if target.is_dir(): shutil.rmtree(target, ignore_errors=True)
    elif target.is_file(): target.unlink()
print('cleaned up.')
